# Ukiyo-e CycleGAN (Kaggle)

Upload this notebook to [Kaggle](https://www.kaggle.com/code) (**New Notebook → File → Import Notebook**). It does not clone a repo or shell out to `train.py`.

Turn **Internet** on and set the accelerator to **GPU**, then run the cells from the top. Training is 20 epochs and will take more than one session. If the session drops, run the cells from the top again with the latest checkpoint attached. Training continues from `latest.pt`: the next epoch if the file is from a finished epoch, or the rest of the interrupted epoch if it was saved partway through.

The checkpoint path is the same as the Colab notebook: `drive/MyDrive/ukiyoe-cyclegan/checkpoints/ukiyoe2photo/latest.pt`, with `loss_log.txt` written beside it. Kaggle cannot mount Google Drive, so this file keeps that path under `/kaggle/working`. To resume the Colab run, add a dataset that contains `checkpoints/ukiyoe2photo/latest.pt` and `loss_log.txt` from `MyDrive/ukiyoe-cyclegan/`. The first cell copies them in. After a run, put the new `latest.pt` and `loss_log.txt` back in that Drive folder. Download them from the notebook output before the session ends; a killed session does not keep `/kaggle/working`.

The generator, discriminator, dataset, and training step below are the CycleGAN pieces from [pytorch-CycleGAN-and-pix2pix](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix): a 9-block ResNet generator, a 70×70 PatchGAN, and a least-squares GAN loss. Domain A is Ukiyo-e, domain B is landscape photos, and direction `BtoA` turns a photo into Ukiyo-e.

`SAVES_PER_EPOCH` in the first code cell is how many times each epoch writes the checkpoint and `loss_log.txt` together. Only the newest checkpoint is kept. Every 100 iterations the training cell prints the last five stat lines and overwrites `checkpoints/ukiyoe2photo/loss_plot.png` with the full loss history. The last cells plot the losses and show 8 test photos, each with its Ukiyo-e translation and its reconstruction.


In [ ]:
import shutil
from pathlib import Path
from types import SimpleNamespace

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Turn on a GPU accelerator first: Settings → Accelerator → GPU.")

device = torch.device("cuda:0")
print(torch.cuda.get_device_name(0))

# Same folders as the Colab notebook. Kaggle cannot mount Drive, so this copy
# lives under the notebook output and is resumed from an attached input.
DRIVE = Path("/kaggle/working/drive/MyDrive/ukiyoe-cyclegan")
CHECKPOINTS = DRIVE / "checkpoints" / "ukiyoe2photo"
RESULTS = DRIVE / "results" / "ukiyoe2photo" / "test_latest" / "images"
DATASET = Path("/tmp/datasets/ukiyoe2photo")
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)


def adopt_saved_checkpoint(checkpoints: Path) -> None:
    destination = checkpoints / "latest.pt"
    if destination.is_file():
        print(f"Using checkpoint at {destination}")
        return
    inputs = Path("/kaggle/input")
    found = []
    if inputs.is_dir():
        found = list(inputs.glob("**/checkpoints/ukiyoe2photo/latest.pt"))
        if not found:
            found = list(inputs.glob("**/latest.pt"))
    if not found:
        print(f"No checkpoint at {destination}. Training starts at epoch 1.")
        return
    source = max(found, key=lambda path: path.stat().st_mtime).parent
    for name in ("latest.pt", "loss_log.txt"):
        src = source / name
        if src.is_file():
            shutil.copy2(src, checkpoints / name)
            print(f"Copied {src} -> {checkpoints / name}")


adopt_saved_checkpoint(CHECKPOINTS)

# Constant learning rate, then linear decay. 10 + 10 = 20 epochs.
N_EPOCHS = 10
N_EPOCHS_DECAY = 10
# How many times to save during each epoch, spread evenly, including the end of the epoch.
SAVES_PER_EPOCH = 3

opt = SimpleNamespace(
    dataroot=str(DATASET),
    phase="train",
    direction="BtoA",
    input_nc=3,
    output_nc=3,
    ngf=64,
    ndf=64,
    netG="resnet_9blocks",
    netD="basic",
    n_layers_D=3,
    norm="instance",
    init_type="normal",
    init_gain=0.02,
    no_dropout=True,
    serial_batches=False,
    batch_size=1,
    load_size=286,
    crop_size=256,
    max_dataset_size=float("inf"),
    preprocess="resize_and_crop",
    no_flip=False,
    epoch_count=1,
    n_epochs=N_EPOCHS,
    n_epochs_decay=N_EPOCHS_DECAY,
    beta1=0.5,
    lr=0.0002,
    gan_mode="lsgan",
    pool_size=50,
    lr_policy="linear",
    lambda_A=10.0,
    lambda_B=10.0,
    lambda_identity=0.5,
    isTrain=True,
    print_freq=100,
    saves_per_epoch=SAVES_PER_EPOCH,
    device=device,
)
print(CHECKPOINTS)


In [ ]:
import zipfile
from urllib.request import urlretrieve

url = "http://efrosgans.eecs.berkeley.edu/cyclegan/datasets/ukiyoe2photo.zip"
zip_path = DATASET.parent / "ukiyoe2photo.zip"

if (DATASET / "trainA").is_dir() and (DATASET / "trainB").is_dir():
    print(f"Using existing dataset at {DATASET}")
else:
    DATASET.parent.mkdir(parents=True, exist_ok=True)

    def report(block, block_size, total):
        if total > 0 and block % 200 == 0:
            print(f"{min(block * block_size, total) / 1e6:.0f} / {total / 1e6:.0f} MB")

    print(f"Downloading {url}")
    urlretrieve(url, zip_path, reporthook=report)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATASET.parent)
    zip_path.unlink()
    print(f"Extracted to {DATASET}")


In [ ]:
import functools

import torch
import torch.nn as nn
from torch.nn import init
from torch.optim import lr_scheduler


class Identity(nn.Module):
    def forward(self, x):
        return x


def get_norm_layer(norm_type="instance"):
    if norm_type == "batch":
        return functools.partial(nn.BatchNorm2d, affine=True, track_running_stats=True)
    if norm_type == "instance":
        return functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)
    if norm_type == "none":
        def norm_layer(x):
            return Identity()
        return norm_layer
    raise NotImplementedError(f"normalization layer [{norm_type}] is not found")


def get_scheduler(optimizer, opt):
    """Keep the learning rate for n_epochs, then decay it to zero over n_epochs_decay."""

    def lambda_rule(epoch):
        return 1.0 - max(0, epoch + opt.epoch_count - opt.n_epochs) / float(opt.n_epochs_decay + 1)

    return lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_rule)


def init_weights(net, init_type="normal", init_gain=0.02):
    def init_func(module):
        classname = module.__class__.__name__
        if hasattr(module, "weight") and (classname.find("Conv") != -1 or classname.find("Linear") != -1):
            if init_type == "normal":
                init.normal_(module.weight.data, 0.0, init_gain)
            elif init_type == "xavier":
                init.xavier_normal_(module.weight.data, gain=init_gain)
            elif init_type == "kaiming":
                init.kaiming_normal_(module.weight.data, a=0, mode="fan_in")
            elif init_type == "orthogonal":
                init.orthogonal_(module.weight.data, gain=init_gain)
            else:
                raise NotImplementedError(f"initialization method [{init_type}] is not implemented")
            if hasattr(module, "bias") and module.bias is not None:
                init.constant_(module.bias.data, 0.0)
        elif classname.find("BatchNorm2d") != -1:
            init.normal_(module.weight.data, 1.0, init_gain)
            init.constant_(module.bias.data, 0.0)

    print(f"initialize network with {init_type}")
    net.apply(init_func)


def init_net(net, device, init_type="normal", init_gain=0.02):
    net.to(device)
    init_weights(net, init_type, init_gain=init_gain)
    return net


class GANLoss(nn.Module):
    def __init__(self, gan_mode, target_real_label=1.0, target_fake_label=0.0):
        super().__init__()
        self.register_buffer("real_label", torch.tensor(target_real_label))
        self.register_buffer("fake_label", torch.tensor(target_fake_label))
        self.gan_mode = gan_mode
        if gan_mode == "lsgan":
            self.loss = nn.MSELoss()
        elif gan_mode == "vanilla":
            self.loss = nn.BCEWithLogitsLoss()
        else:
            raise NotImplementedError(f"gan mode {gan_mode} not implemented")

    def __call__(self, prediction, target_is_real):
        target = self.real_label if target_is_real else self.fake_label
        return self.loss(prediction, target.expand_as(prediction))


class ResnetBlock(nn.Module):
    def __init__(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        super().__init__()
        self.conv_block = self.build_conv_block(dim, padding_type, norm_layer, use_dropout, use_bias)

    def build_conv_block(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        block = []
        padding = 0
        if padding_type == "reflect":
            block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            padding = 1
        else:
            raise NotImplementedError(f"padding [{padding_type}] is not implemented")
        block += [nn.Conv2d(dim, dim, kernel_size=3, padding=padding, bias=use_bias), norm_layer(dim), nn.ReLU(True)]
        if use_dropout:
            block += [nn.Dropout(0.5)]

        padding = 0
        if padding_type == "reflect":
            block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            padding = 1
        else:
            raise NotImplementedError(f"padding [{padding_type}] is not implemented")
        block += [nn.Conv2d(dim, dim, kernel_size=3, padding=padding, bias=use_bias), norm_layer(dim)]
        return nn.Sequential(*block)

    def forward(self, x):
        return x + self.conv_block(x)


class ResnetGenerator(nn.Module):
    def __init__(self, input_nc, output_nc, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False, n_blocks=9, padding_type="reflect"):
        assert n_blocks >= 0
        super().__init__()
        if isinstance(norm_layer, functools.partial):
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d

        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=use_bias),
            norm_layer(ngf),
            nn.ReLU(True),
        ]
        n_downsampling = 2
        for i in range(n_downsampling):
            mult = 2**i
            model += [
                nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=use_bias),
                norm_layer(ngf * mult * 2),
                nn.ReLU(True),
            ]
        mult = 2**n_downsampling
        for _ in range(n_blocks):
            model += [ResnetBlock(ngf * mult, padding_type=padding_type, norm_layer=norm_layer, use_dropout=use_dropout, use_bias=use_bias)]
        for i in range(n_downsampling):
            mult = 2 ** (n_downsampling - i)
            model += [
                nn.ConvTranspose2d(ngf * mult, int(ngf * mult / 2), kernel_size=3, stride=2, padding=1, output_padding=1, bias=use_bias),
                norm_layer(int(ngf * mult / 2)),
                nn.ReLU(True),
            ]
        model += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


class NLayerDiscriminator(nn.Module):
    def __init__(self, input_nc, ndf=64, n_layers=3, norm_layer=nn.BatchNorm2d):
        super().__init__()
        if isinstance(norm_layer, functools.partial):
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d

        sequence = [
            nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),
        ]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2**n, 8)
            sequence += [
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=4, stride=2, padding=1, bias=use_bias),
                norm_layer(ndf * nf_mult),
                nn.LeakyReLU(0.2, True),
            ]
        nf_mult_prev = nf_mult
        nf_mult = min(2**n_layers, 8)
        sequence += [
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=4, stride=1, padding=1, bias=use_bias),
            norm_layer(ndf * nf_mult),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * nf_mult, 1, kernel_size=4, stride=1, padding=1),
        ]
        self.model = nn.Sequential(*sequence)

    def forward(self, x):
        return self.model(x)


def define_G(input_nc, output_nc, ngf, netG, norm="instance", use_dropout=False):
    norm_layer = get_norm_layer(norm)
    if netG == "resnet_9blocks":
        return ResnetGenerator(input_nc, output_nc, ngf, norm_layer=norm_layer, use_dropout=use_dropout, n_blocks=9)
    raise NotImplementedError(f"Generator model name [{netG}] is not recognized")


def define_D(input_nc, ndf, netD, n_layers_D=3, norm="instance"):
    norm_layer = get_norm_layer(norm)
    if netD == "basic":
        return NLayerDiscriminator(input_nc, ndf, n_layers=3, norm_layer=norm_layer)
    if netD == "n_layers":
        return NLayerDiscriminator(input_nc, ndf, n_layers_D, norm_layer=norm_layer)
    raise NotImplementedError(f"Discriminator model name [{netD}] is not recognized")


In [ ]:
import random
from pathlib import Path

import torch.utils.data as data
import torchvision.transforms as transforms
from PIL import Image

IMG_EXTENSIONS = [".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".tif", ".tiff"]


def make_dataset(directory, max_dataset_size=float("inf")):
    root = Path(directory)
    if not root.is_dir():
        raise FileNotFoundError(f"{directory} is not a valid directory")
    images = [
        str(path)
        for path in sorted(root.rglob("*"))
        if path.is_file() and path.suffix.lower() in IMG_EXTENSIONS
    ]
    return images[: min(max_dataset_size, len(images))]


def get_transform(opt):
    transform = []
    if "resize" in opt.preprocess:
        transform.append(transforms.Resize([opt.load_size, opt.load_size], transforms.InterpolationMode.BICUBIC))
    if "crop" in opt.preprocess:
        transform.append(transforms.RandomCrop(opt.crop_size))
    if not opt.no_flip:
        transform.append(transforms.RandomHorizontalFlip())
    transform += [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
    return transforms.Compose(transform)


class UnalignedDataset(data.Dataset):
    """Unpaired images from trainA/trainB or testA/testB."""

    def __init__(self, opt):
        root = Path(opt.dataroot)
        self.A_paths = make_dataset(root / f"{opt.phase}A", opt.max_dataset_size)
        self.B_paths = make_dataset(root / f"{opt.phase}B", opt.max_dataset_size)
        self.A_size = len(self.A_paths)
        self.B_size = len(self.B_paths)
        self.opt = opt
        self.transform_A = get_transform(opt)
        self.transform_B = get_transform(opt)

    def __getitem__(self, index):
        A_path = self.A_paths[index % self.A_size]
        if self.opt.serial_batches:
            index_B = index % self.B_size
        else:
            index_B = random.randint(0, self.B_size - 1)
        B_path = self.B_paths[index_B]
        A = self.transform_A(Image.open(A_path).convert("RGB"))
        B = self.transform_B(Image.open(B_path).convert("RGB"))
        return {"A": A, "B": B, "A_paths": A_path, "B_paths": B_path}

    def __len__(self):
        return max(self.A_size, self.B_size)


In [ ]:
import itertools
import random

import torch


class ImagePool:
    """Buffer of previously generated images, used to update the discriminators."""

    def __init__(self, pool_size):
        self.pool_size = pool_size
        if pool_size > 0:
            self.num_imgs = 0
            self.images = []

    def query(self, images):
        if self.pool_size == 0:
            return images
        return_images = []
        for image in images:
            image = torch.unsqueeze(image.data, 0)
            if self.num_imgs < self.pool_size:
                self.num_imgs += 1
                self.images.append(image)
                return_images.append(image)
            else:
                if random.uniform(0, 1) > 0.5:
                    random_id = random.randint(0, self.pool_size - 1)
                    tmp = self.images[random_id].clone()
                    self.images[random_id] = image
                    return_images.append(tmp)
                else:
                    return_images.append(image)
        return torch.cat(return_images, 0)


def set_requires_grad(nets, requires_grad):
    if not isinstance(nets, list):
        nets = [nets]
    for net in nets:
        for param in net.parameters():
            param.requires_grad = requires_grad


class CycleGANModel:
    """G_A: A→B, G_B: B→A. With direction BtoA, real_A is a photo and fake_B is Ukiyo-e."""

    def __init__(self, opt):
        self.opt = opt
        self.device = opt.device
        self.isTrain = opt.isTrain
        self.loss_names = ["D_A", "G_A", "cycle_A", "idt_A", "D_B", "G_B", "cycle_B", "idt_B"]
        self.visual_names = ["real_A", "fake_B", "rec_A", "real_B", "fake_A", "rec_B"]
        self.model_names = ["G_A", "G_B", "D_A", "D_B"] if self.isTrain else ["G_A", "G_B"]
        self.save_dir = CHECKPOINTS

        self.netG_A = define_G(opt.input_nc, opt.output_nc, opt.ngf, opt.netG, opt.norm, use_dropout=not opt.no_dropout)
        self.netG_B = define_G(opt.output_nc, opt.input_nc, opt.ngf, opt.netG, opt.norm, use_dropout=not opt.no_dropout)
        init_net(self.netG_A, self.device, opt.init_type, opt.init_gain)
        init_net(self.netG_B, self.device, opt.init_type, opt.init_gain)

        if self.isTrain:
            self.netD_A = define_D(opt.output_nc, opt.ndf, opt.netD, opt.n_layers_D, opt.norm)
            self.netD_B = define_D(opt.input_nc, opt.ndf, opt.netD, opt.n_layers_D, opt.norm)
            init_net(self.netD_A, self.device, opt.init_type, opt.init_gain)
            init_net(self.netD_B, self.device, opt.init_type, opt.init_gain)
            if opt.lambda_identity > 0:
                assert opt.input_nc == opt.output_nc
            self.fake_A_pool = ImagePool(opt.pool_size)
            self.fake_B_pool = ImagePool(opt.pool_size)
            self.criterionGAN = GANLoss(opt.gan_mode).to(self.device)
            self.criterionCycle = torch.nn.L1Loss()
            self.criterionIdt = torch.nn.L1Loss()
            self.optimizer_G = torch.optim.Adam(
                itertools.chain(self.netG_A.parameters(), self.netG_B.parameters()),
                lr=opt.lr,
                betas=(opt.beta1, 0.999),
            )
            self.optimizer_D = torch.optim.Adam(
                itertools.chain(self.netD_A.parameters(), self.netD_B.parameters()),
                lr=opt.lr,
                betas=(opt.beta1, 0.999),
            )
            self.optimizers = [self.optimizer_G, self.optimizer_D]
            self.schedulers = [get_scheduler(optimizer, opt) for optimizer in self.optimizers]
        self.print_networks()

    def print_networks(self):
        print("---------- Networks initialized -------------")
        for name in self.model_names:
            net = getattr(self, "net" + name)
            num_params = sum(param.numel() for param in net.parameters())
            print(f"[Network {name}] Total number of parameters : {num_params / 1e6:.3f} M")
        print("-----------------------------------------------")

    def set_input(self, data):
        AtoB = self.opt.direction == "AtoB"
        self.real_A = data["A" if AtoB else "B"].to(self.device)
        self.real_B = data["B" if AtoB else "A"].to(self.device)
        self.image_paths = data["A_paths" if AtoB else "B_paths"]

    def forward(self):
        self.fake_B = self.netG_A(self.real_A)
        self.rec_A = self.netG_B(self.fake_B)
        self.fake_A = self.netG_B(self.real_B)
        self.rec_B = self.netG_A(self.fake_A)

    def backward_D_basic(self, netD, real, fake):
        loss_D = (self.criterionGAN(netD(real), True) + self.criterionGAN(netD(fake.detach()), False)) * 0.5
        loss_D.backward()
        return loss_D

    def backward_G(self):
        lambda_idt = self.opt.lambda_identity
        lambda_A = self.opt.lambda_A
        lambda_B = self.opt.lambda_B
        if lambda_idt > 0:
            self.idt_A = self.netG_A(self.real_B)
            self.loss_idt_A = self.criterionIdt(self.idt_A, self.real_B) * lambda_B * lambda_idt
            self.idt_B = self.netG_B(self.real_A)
            self.loss_idt_B = self.criterionIdt(self.idt_B, self.real_A) * lambda_A * lambda_idt
        else:
            self.loss_idt_A = 0
            self.loss_idt_B = 0
        self.loss_G_A = self.criterionGAN(self.netD_A(self.fake_B), True)
        self.loss_G_B = self.criterionGAN(self.netD_B(self.fake_A), True)
        self.loss_cycle_A = self.criterionCycle(self.rec_A, self.real_A) * lambda_A
        self.loss_cycle_B = self.criterionCycle(self.rec_B, self.real_B) * lambda_B
        self.loss_G = (
            self.loss_G_A + self.loss_G_B + self.loss_cycle_A + self.loss_cycle_B + self.loss_idt_A + self.loss_idt_B
        )
        self.loss_G.backward()

    def optimize_parameters(self):
        self.forward()
        set_requires_grad([self.netD_A, self.netD_B], False)
        self.optimizer_G.zero_grad()
        self.backward_G()
        self.optimizer_G.step()
        set_requires_grad([self.netD_A, self.netD_B], True)
        self.optimizer_D.zero_grad()
        self.loss_D_A = self.backward_D_basic(self.netD_A, self.real_B, self.fake_B_pool.query(self.fake_B))
        self.loss_D_B = self.backward_D_basic(self.netD_B, self.real_A, self.fake_A_pool.query(self.fake_A))
        self.optimizer_D.step()

    def get_current_losses(self):
        return {name: float(getattr(self, "loss_" + name)) for name in self.loss_names}

    def update_learning_rate(self):
        old_lr = self.optimizers[0].param_groups[0]["lr"]
        for scheduler in self.schedulers:
            scheduler.step()
        lr = self.optimizers[0].param_groups[0]["lr"]
        print(f"learning rate {old_lr:.7f} -> {lr:.7f}")

    def save_checkpoint(self, epoch, batches_done, total_iters, loss_history, log_text, epoch_complete):
        path = self.save_dir / "latest.pt"
        torch.save(
            {
                "epoch": epoch,
                "batches_done": batches_done,
                "epoch_complete": epoch_complete,
                "total_iters": total_iters,
                "n_epochs": self.opt.n_epochs,
                "n_epochs_decay": self.opt.n_epochs_decay,
                "nets": {name: getattr(self, "net" + name).state_dict() for name in self.model_names},
                "optimizers": [optimizer.state_dict() for optimizer in self.optimizers],
                "schedulers": [scheduler.state_dict() for scheduler in self.schedulers],
                "loss_history": loss_history,
                "log_text": log_text,
            },
            path,
        )
        for old in self.save_dir.glob("*_net_*.pth"):
            old.unlink()
        where = "end" if epoch_complete else f"batch {batches_done}"
        print(f"saved {path} at {where} of epoch {epoch}")


In [ ]:
import time

import matplotlib.pyplot as plt
import IPython.display as ipython_display
from torch.utils.data import DataLoader

LOSS_PLOT = CHECKPOINTS / "loss_plot.png"
loss_plot_view = None


def draw_loss_figure(loss_history, loss_names):
    figure, axes = plt.subplots(
        len(loss_names), 1, figsize=(14, 3.5 * len(loss_names)), sharex=True, layout="constrained"
    )
    if len(loss_names) == 1:
        axes = [axes]
    for axis, name in zip(axes, loss_names):
        axis.plot(loss_history[name], linewidth=1.5)
        axis.set_title(name, fontsize=14, loc="left")
        axis.set_xlabel("log step", fontsize=12)
        axis.tick_params(labelsize=11)
    figure.suptitle("Training losses, logged every 100 iterations", fontsize=16)
    return figure


def update_loss_plot(loss_history, loss_names):
    """Overwrite loss_plot.png and refresh the one inline image."""
    global loss_plot_view
    figure = draw_loss_figure(loss_history, loss_names)
    figure.savefig(LOSS_PLOT)
    plt.close(figure)
    image = ipython_display.Image(data=LOSS_PLOT.read_bytes())
    if loss_plot_view is None:
        loss_plot_view = ipython_display.display(image, display_id=True)
    else:
        loss_plot_view.update(image)


def iteration_stats(lines):
    return [line for line in lines if line.startswith("(epoch:")]

dataset = UnalignedDataset(opt)
loader = DataLoader(dataset, batch_size=opt.batch_size, shuffle=not opt.serial_batches, num_workers=0)
print(f"The number of training images = {len(dataset)}")

def checkpoint_save_points(dataset_size, saves_per_epoch):
    if saves_per_epoch < 1:
        raise ValueError("SAVES_PER_EPOCH must be at least 1")
    return sorted({(dataset_size * i) // saves_per_epoch for i in range(1, saves_per_epoch + 1)})


model = CycleGANModel(opt)
loss_history = {name: [] for name in model.loss_names}
log_path = CHECKPOINTS / "loss_log.txt"
checkpoint_path = CHECKPOINTS / "latest.pt"
total_iters = 0
start_epoch = 1
resume_batches = 0
epochs = opt.n_epochs + opt.n_epochs_decay
dataset_size = len(dataset)
save_points = checkpoint_save_points(dataset_size, opt.saves_per_epoch)
log_lines = ["================ Training Loss ================"]

if checkpoint_path.is_file():
    try:
        checkpoint = torch.load(checkpoint_path, map_location=opt.device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=opt.device)
    saved_epochs = checkpoint["n_epochs"] + checkpoint["n_epochs_decay"]
    if saved_epochs != epochs:
        raise RuntimeError(
            f"{checkpoint_path} is from a {saved_epochs}-epoch run. This notebook trains for {epochs} epochs. Delete that file to start over."
        )
    for name, state in checkpoint["nets"].items():
        getattr(model, "net" + name).load_state_dict(state)
    for optimizer, state in zip(model.optimizers, checkpoint["optimizers"]):
        optimizer.load_state_dict(state)
    for scheduler, state in zip(model.schedulers, checkpoint["schedulers"]):
        scheduler.load_state_dict(state)
    loss_history = checkpoint["loss_history"]
    total_iters = checkpoint["total_iters"]
    # Older checkpoints were written only at the end of an epoch and have no epoch_complete flag.
    epoch_complete = checkpoint.get("epoch_complete", True)
    if epoch_complete:
        start_epoch = checkpoint["epoch"] + 1
        resume_batches = 0
        print(f"Resuming at epoch {start_epoch} / {epochs} from {checkpoint_path}")
    else:
        start_epoch = checkpoint["epoch"]
        resume_batches = checkpoint["batches_done"]
        print(
            f"Resuming epoch {start_epoch} / {epochs} at batch {resume_batches} / {dataset_size} from {checkpoint_path}"
        )
    if "log_text" in checkpoint:
        log_lines = checkpoint["log_text"].splitlines()
    elif log_path.is_file():
        log_lines = log_path.read_text().splitlines()
else:
    print(f"No checkpoint at {checkpoint_path}. Starting at epoch 1 / {epochs}")

print(f"Saving at batches {save_points} of each {dataset_size}-step epoch")

if loss_history["D_A"]:
    update_loss_plot(loss_history, model.loss_names)

if start_epoch > epochs:
    print(f"Training already finished ({epochs} epochs).")
else:
    for epoch in range(start_epoch, epochs + 1):
        epoch_start = time.time()
        batches_done = resume_batches if epoch == start_epoch else 0
        resume_batches = 0
        remaining = dataset_size - batches_done
        taken = 0
        for data in loader:
            if taken >= remaining:
                break
            taken += 1
            batches_done += 1
            total_iters += opt.batch_size
            model.set_input(data)
            model.optimize_parameters()
            if total_iters % opt.print_freq == 0:
                losses = model.get_current_losses()
                for name, value in losses.items():
                    loss_history[name].append(value)
                message = f"(epoch: {epoch}, iters: {total_iters}) " + " ".join(
                    f"{name}: {value:.3f}" for name, value in losses.items()
                )
                log_lines.append(message)
                for line in iteration_stats(log_lines)[-5:]:
                    print(line, flush=True)
                update_loss_plot(loss_history, model.loss_names)
            if batches_done in save_points and batches_done != dataset_size:
                log_text = "\n".join(log_lines) + "\n"
                model.save_checkpoint(epoch, batches_done, total_iters, loss_history, log_text, False)
                log_path.write_text(log_text)
        model.update_learning_rate()
        log_text = "\n".join(log_lines) + "\n"
        model.save_checkpoint(epoch, batches_done, total_iters, loss_history, log_text, True)
        log_path.write_text(log_text)
        print(f"End of epoch {epoch} / {epochs} \t Time Taken: {time.time() - epoch_start:.0f} sec", flush=True)


In [ ]:
import matplotlib.pyplot as plt

if not loss_history["D_A"]:
    raise RuntimeError(f"No losses recorded. Check {log_path}")

draw_loss_figure(loss_history, model.loss_names)
plt.show()


In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PilImage
from torch.utils.data import DataLoader


def tensor_to_image(tensor):
    image = tensor[0].detach().cpu().float().numpy()
    image = (np.transpose(image, (1, 2, 0)) + 1) / 2.0 * 255.0
    return image.astype(np.uint8)


def batch_path(value):
    return value[0] if isinstance(value, (list, tuple)) else value


test_opt = copy.copy(opt)
test_opt.phase = "test"
test_opt.serial_batches = True
test_opt.no_flip = True
test_opt.load_size = opt.crop_size
test_set = UnalignedDataset(test_opt)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, num_workers=0)

model.netG_A.eval()
model.netG_B.eval()
rows = []
for index, batch in enumerate(test_loader):
    if index >= 8:
        break
    model.set_input(batch)
    with torch.no_grad():
        model.forward()
    stem = Path(batch_path(model.image_paths)).stem
    panels = []
    for label in ("real_A", "fake_B", "rec_A"):
        image = tensor_to_image(getattr(model, label))
        PilImage.fromarray(image).save(RESULTS / f"{stem}_{label}.png")
        panels.append(image)
    rows.append(panels)

figure, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
if len(rows) == 1:
    axes = axes.reshape(1, 3)
titles = ["photo", "ukiyo-e", "reconstructed photo"]
for row, panels in enumerate(rows):
    for column, image in enumerate(panels):
        axes[row, column].imshow(image)
        axes[row, column].set_title(titles[column])
        axes[row, column].axis("off")
figure.tight_layout()
plt.show()
